# 21. Dartboard Retrieval

Pick results that jointly cover the query and each other - relevance *and* diversity in one score.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Top-k often returns five near-identical chunks. **Dartboard retrieval** treats result selection as covering a dartboard: pick a *set* of passages such that wherever the 'dart' (the real answer) lands, one selected passage is nearby, while staying close to the query. It scores candidates by Gaussian-smoothed relevance **and** coverage of the rest of the pool, balancing both with two weights. (This notebook uses a simplified greedy version.)

## How it works

1. Retrieve a pool of candidates
2. Compute pairwise distances and query distances
3. Convert distances to log-Gaussian closeness
4. Greedily add the candidate that maximises relevance + coverage
5. Update coverage and repeat for k picks

```text
 pool of 20 --> pick #1 closest to query --> pick #2 = relevant AND covers new area --> ... k picks
 relevance_weight vs diversity_weight controls the balance
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

Dartboard (simplified): think of throwing darts at the query. A good *set* of results is one where, wherever the dart lands, some selected passage is close. Greedy selection maximises the Gaussian-smoothed coverage of the candidate pool.

In [ ]:
import numpy as np

def dartboard(query, k=4, pool=20, sigma=0.4, div_weight=1.0, rel_weight=1.0):
    q = embed(query)[0]
    ids = [h["idx"] for h in store.search(query, pool)]
    V = store.vecs[ids]
    dist = 1 - V @ V.T                         # pairwise cosine distance
    qdist = 1 - V @ q
    log_p = -(dist ** 2) / (2 * sigma ** 2)    # log of Gaussian "closeness"
    chosen = []
    best_cover = np.full(len(ids), -np.inf)    # best log-prob of each candidate being covered by a chosen doc
    for _ in range(k):
        scores = []
        for i in range(len(ids)):
            if i in chosen:
                scores.append(-np.inf); continue
            cover = np.maximum(best_cover, log_p[i])
            scores.append(rel_weight * (-(qdist[i] ** 2) / (2 * sigma ** 2)) + div_weight * np.logaddexp.reduce(cover))
        pick = int(np.argmax(scores))
        chosen.append(pick)
        best_cover = np.maximum(best_cover, log_p[pick])
    return [{"idx": ids[i], "text": chunks[ids[i]], "score": float(1 - qdist[i]), "meta": {}} for i in chosen]

query = "How can I make a RAG system more accurate?"
print("--- top-k"); show(store.search(query, 4), width=90)
print("--- dartboard"); show(dartboard(query), width=90)

In [ ]:
print(answer(query, [h["text"] for h in dartboard(query)]))

Tune `div_weight` up for broader coverage, down for a tight match to the query.

## Quantify diversity

In [ ]:
import numpy as np
def redundancy(hits):
    v = store.vecs[[h["idx"] for h in hits]]
    return float(((v @ v.T).sum() - len(hits)) / (len(hits) * (len(hits) - 1)))

q = "How can I make a RAG system more accurate?"
print("top-k    redundancy:", round(redundancy(store.search(q, 4)), 3))
print("dartboard redundancy:", round(redundancy(dartboard(q)), 3))

## Sweep the diversity weight

In [ ]:
import pandas as pd
rows = []
for dw in [0.0, 0.5, 1.0, 2.0, 4.0]:
    hits = dartboard(q, div_weight=dw)
    rows.append({"div_weight": dw, "redundancy": round(redundancy(hits), 3), "avg_relevance": round(float(np.mean([h["score"] for h in hits])), 3)})
pd.DataFrame(rows).set_index("div_weight")

## Evaluate

In [ ]:
compare_retrievers({"top-k": store.search, "dartboard": lambda q, k=3: dartboard(q, k=k)}, k=3)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Balances relevance and diversity in one objective | More compute than top-k |
| Fewer redundant chunks | Needs a candidate pool |
| Tunable weights | Weights/sigma must be tuned |

## When to use it

Broad or exploratory questions where several perspectives help, and top-k keeps returning the same paragraph.

## Experiments to try

- Tune `sigma` and watch coverage change.
- Compare to MMR (notebook 19) on redundancy and hit rate.
- Increase the pool to 50.

## Key takeaways

- Select sets, not just individual results.
- Redundancy lowers the value of every extra chunk.
- Weights express your relevance/diversity preference.

---
**Next:** [22. Multimodal RAG with Captioning](../22_multi_model_rag_with_captioning/multi_model_rag_with_captioning.ipynb)